In [13]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [14]:
import pbn_37k
from pbn_37k.ask import sigAssistant

import pandas as pd
from tqdm import tqdm
tqdm.pandas()

from dotenv import load_dotenv
load_dotenv()

brain = sigAssistant(pathCache=".cache/", llm_highend="gpt-4o",  llm_fast="gpt-4o-mini")

In [15]:
with open("data/cities/prompt/prompt.md", "r") as f:
    prompt_template = f.read()
with open("data/cities/activities.md", "r") as f:
    prompt_activity = f.read()
with open("data/cities/portfolio.md", "r") as f:
    prompt_portfolio = f.read()
with open("data/cities/prompt/cities.md", "r") as f:
    cities = [x.strip("*").strip() for x in f.read().split("\n") ]
cn = ["gotham", "baraddur", "kingslanding", "coruscant", "wakanda", "shire", "capitol"]

In [16]:
HEADER = """---
layout: default
title: TITLE
parent: PARENT
has_children: true
nav_order: NAVORDER
---\n\n"""

In [17]:
CITIES = {}
for i in range(len(cities)):
    desc = cities[i]
    q = prompt_template.replace("$CITYNAME$", desc)
    urban_report = brain.ask(q)
    for k in range(3):
        urban_report = urban_report.replace("Page "+str(k+1)+": ", "").replace("---","")
    report = HEADER.replace("TITLE", cn[i].title()).replace("NAVORDER", str(i+3)).replace("\nparent: PARENT","") + urban_report
    CITIES[cn[i]] = {"report": report}
    
    with open(f"docs/{cn[i]}_report.md", "w") as f:
        f.write(report)

--- gpt-4o-mini:	 2025-10-23 00:15:25
--- gpt-4o-mini:	 2025-10-23 00:15:25
--- gpt-4o-mini:	 2025-10-23 00:15:25
--- gpt-4o-mini:	 2025-10-23 00:15:25
--- gpt-4o-mini:	 2025-10-23 00:15:25
--- gpt-4o-mini:	 2025-10-23 00:15:25
--- gpt-4o-mini:	 2025-10-23 00:15:25


In [ ]:
for i in range(len(cities)):
    desc = CITIES[cn[i]]["report"]
    q = prompt_portfolio.replace("DESCRIPTION", desc)
    urban_report = brain.ask(q)
    for k in range(3):
        urban_report = urban_report.replace("Page "+str(k+1)+": ", "").replace("---","")
    report = HEADER.replace("TITLE", cn[i].title()+"'s portfolio").replace("NAVORDER", str(i+1)).replace("PARENT",cn[i].title()) + urban_report
    CITIES[cn[i]]["portfolio"] = {"portfolio": report}
    with open(f"docs/{cn[i]}_portfolio.md", "w") as f:
        f.write(report)

    for k in range(7):
        q = prompt_activity.replace("$$DESC$$", desc).replace("$$ACT$$", urban_report)
        q = q.replace("$$NN$$ ", str(k+1))
        activity_report = brain.ask(q)

        reportB = HEADER.replace("TITLE", cn[i].title()+f"'s activity {k+1}").replace("NAVORDER", str(k+2)).replace("PARENT",cn[i].title()) + activity_report
        reportB = report.replace("has_children: true _n","")
        CITIES[cn[i]]["portfolio"][f"activity_{k+1}"] = reportB
        with open(f"docs/{cn[i]}_activity_{k+1}.md", "w") as f:
            f.write(reportB)


    


In [ ]:
%pip freeze > requirements.txt

Note: you may need to restart the kernel to use updated packages.
